In [ ]:
import torch
import torch.nn as nn

In [ ]:
# Autoencoder Same as Embedding
class Autoencoder(nn.Module):
    def __init__(self, in_dim=25, bottleneck_dim=3, hidden_dim=512):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(in_dim, int(hidden_dim/4)),
            nn.ReLU(),
            nn.LayerNorm(int(hidden_dim/4)),
            nn.Dropout(0.1),
            nn.Linear(int(hidden_dim/4), int(hidden_dim/2)),
            nn.ReLU(),
            nn.LayerNorm(int(hidden_dim/2)),
            nn.Dropout(0.1),
            nn.Linear(int(hidden_dim/2), int(hidden_dim)),
            nn.ReLU(),
            nn.LayerNorm(int(hidden_dim)),
            nn.Dropout(0.1),
            nn.Linear(hidden_dim, int(hidden_dim/2)),
            nn.ReLU(),
            nn.LayerNorm(int(hidden_dim/2)),
            nn.Dropout(0.1),
            nn.Linear(int(hidden_dim/2), int(hidden_dim/4)),
            nn.ReLU(),
            nn.LayerNorm(int(hidden_dim/4)),
            nn.Dropout(0.1),
            nn.Linear(int(hidden_dim/4), bottleneck_dim)
        )
        self.decoder = nn.Sequential(
            nn.Linear(bottleneck_dim, int(hidden_dim/4)),
            nn.ReLU(),
            nn.LayerNorm(int(hidden_dim/4)),
            nn.Dropout(0.1),
            nn.Linear(int(hidden_dim/4), int(hidden_dim/2)),
            nn.ReLU(),
            nn.LayerNorm(int(hidden_dim/2)),
            nn.Dropout(0.1),
            nn.Linear(int(hidden_dim/2), hidden_dim),
            nn.ReLU(),
            nn.LayerNorm(int(hidden_dim)),
            nn.Dropout(0.1),
            nn.Linear(hidden_dim, int(hidden_dim/2)),
            nn.ReLU(),
            nn.LayerNorm(int(hidden_dim/2)),
            nn.Dropout(0.1),
            nn.Linear(int(hidden_dim/2), int(hidden_dim/4)),
            nn.ReLU(),
            nn.LayerNorm(int(hidden_dim/4)),
            nn.Dropout(0.1),
            nn.Linear(int(hidden_dim/4), in_dim),
        )

    def forward(self, x):
        z = self.encoder(x)
        return z, self.decoder(z)

In [ ]:
def load_and_recover_z3d_png(path):
    """
    Load a PNG image as RGB, interpret it as normalized z_3d embedding encoded as uint8 [0–255],
    and recover the original z_3d float values via inverse scaling.
    """
    img = Image.open(path).convert("RGB")
    arr = torch.ByteTensor(torch.ByteStorage.from_buffer(img.tobytes()))
    rgb = arr.view(img.size[1], img.size[0], 3).permute(2, 0, 1).clone()  # [3, H, W], uint8
    mask = (rgb > 245).all(dim=0) #block dim color
    rgb[:, mask] = 255
    rgb_float = rgb.float() / 255.0  # [3, H, W], float32
    non_white_pixels = ((rgb_float != 1).any(dim=0)).sum().item()
    print(f"count for non white: {non_white_pixels}")

    return rgb_float

In [ ]:
def infer_cell_map(latent_image, model):
    min_vals = torch.tensor([-69.761505, -75.65188,  -77.16103], device=latent_image.device) #change this according to your autoencoder output scale
    range_vals = torch.tensor([88.969406, 65.244896, 67.13518], device=latent_image.device) - min_vals #change this according to your autoencoder output scale
    H, W = latent_image.shape[1], latent_image.shape[2]

    latent_image = latent_image.to("cuda")
    range_vals = range_vals.to("cuda")
    min_vals = min_vals.to("cuda")

    flat_img = latent_image.permute(1, 2, 0).reshape(-1, 3)
    white_mask = (flat_img == 1.0).all(dim=1)
    infer_input_rgb = flat_img[~white_mask]
    pred = torch.zeros(flat_img.shape[0], dtype=torch.long, device="cuda")
    if infer_input_rgb.shape[0] > 0:
        z_recovered = infer_input_rgb * range_vals + min_vals
        logits = model.decoder(z_recovered)
        pred[~white_mask] = torch.argmax(logits, dim=1) + 1
    pred = pred.reshape(1, H, W)
    return pred

In [ ]:
if __name__ == '__main__':
    model = Autoencoder().to("cuda")
    model.load_state_dict(torch.load("autoencoder_checkpoint_path"))
    model.eval()
    img = load_and_recover_z3d_png("img_path")
    cellmap = infer_cell_map(img, model)